In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.17 BCS Superconductivity

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.17",
    title="BCS Superconductivity",
    blurb="The volume's finale gives the interaction map its attractive "
    "corner. Cooper's instability — any attraction binds a pair at the "
    "Fermi surface, with a binding energy no perturbation series can "
    "see — leads to the BCS gap equation, solved by brentq across the "
    "whole temperature range: T_c = 1.134 ω_D e^(−1/λ) reproduced to "
    "five digits, the universal ratios 2Δ/k_BT_c = 3.53 and "
    "ΔC/C = 1.43 landing on their closed forms, the √(1 − T/T_c) "
    "order-parameter collapse, and the gapped density of states behind "
    "every tunneling spectrum. Judged against real elements: aluminum, "
    "tin, indium within 4% of the universal ratio — and lead, honestly "
    "outside it. The course's last verdict table.",
    difficulty="advanced",
    estimate="150–180 min",
)

## Notebook overview

Every interaction in this volume repelled. The Mott insulator of
[§8.13](hubbard-model.ipynb), the quasiparticle of
[§8.14](quasiparticles-gw.ipynb), the exciton of
[§8.15](optics-excitons.ipynb) — all children of Coulomb repulsion.
But [§8.9](tight-binding.ipynb)'s Peierls calculation already showed
the lattice talking back to the electrons, and the same
electron–phonon handshake that dimerized the chain produces, in three
dimensions, something stranger: a *retarded attraction* between
electrons near the Fermi surface (one electron polarizes the lattice;
a second, arriving later, feels the lingering positive wake). The
consequence — resistance vanishing below a critical temperature,
measured by Kamerlingh Onnes in 1911 — resisted theory for 46 years.
This notebook computes the resolution {cite}`bardeen1957`, the last
summit of the physics volumes.

The logic unfolds in the order history found it. First **Cooper's
instability** {cite}`cooper1956`: two electrons atop a frozen Fermi
sea, attracting weakly within a Debye shell $\omega_D$, *always* bind
— the Fermi surface makes the pairing susceptibility logarithmic, and
the binding energy $2\omega_D e^{-2/\lambda}$ (reproduced by our
`brentq` solution to $0.1\%$ at $\lambda = 0.2$) has an essential
singularity at $\lambda = 0$: no Taylor expansion in the coupling ever
sees it, which is *why* 46 years. Then the full **BCS gap equation**,
solved numerically at every temperature: the zero-temperature gap
$\Delta_0 = \omega_D/\sinh(1/\lambda)$ matched at $10^{-10}$, the
transition temperature $T_c = 1.134\,\omega_D e^{-1/\lambda}$
reproduced to five digits, and — the theory's glory — their *ratio*,
in which every material parameter cancels:
$2\Delta_0/k_BT_c = 3.528$, universal. The order parameter collapses
as $3.06\sqrt{1 - T/T_c}$ near $T_c$ ([§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb)'s
mean-field exponent $\beta = \tfrac12$, found once more at this
volume's closing phase transition); the electronic specific heat jumps by
$\Delta C/C_n = 1.43$ at $T_c$ (computed $1.43$ from a numerical
entropy, against the closed form $12/7\zeta(3)$) and freezes out
exponentially below — the gap acting exactly as
[§7.13](../07-quantum-statistical-mechanics/semiconductor-statistics.ipynb)'s
gaps did, but *self-generated*. The quasiparticle density of states
develops the $E/\sqrt{E^2 - \Delta^2}$ coherence peaks of every
tunneling experiment, conserving states by a closed-form ledger. And
the finale judges the theory against the elements: aluminum, tin, and
indium carry measured $2\Delta/k_BT_c$ within $4\%$ of $3.528$ —
lead, at $4.38$, sits $24\%$ high, and the notebook says why (strong
coupling: the retardation BCS idealizes away) {cite}`tinkham2004`.
The volume ends where physics usually begins: with a theory that
works, and a measured exception that points past it.

> **Conventions (this notebook).** Units $k_B = 1$ and $\hbar = 1$;
> the Debye energy $\omega_D = 1$ sets the scale, and the
> dimensionless coupling is $\lambda = N_0 V$ (density of states at
> the Fermi level times the pairing attraction). Default coupling
> $\lambda = 0.3$ unless swept. Energies $\xi$ are measured from the
> Fermi level; particle–hole symmetry is assumed (integrals written
> over $\xi > 0$ with explicit factors). Gap equations are closed
> with `scipy.optimize.brentq` on `scipy.integrate.quad` integrals;
> entropies use the numerically stable softplus form and specific
> heats central differences with steps *relative to* $T_c$.
>
> **How to read the checks.** Each exercise closes with a `validate`
> call against an independent fact: a closed-form limit, a universal
> ratio, a measured element. A ✓ is strong evidence; a ✗ is a prompt
> to locate the discrepancy, not an automatic verdict.
>
> **Scope.** Weak-coupling s-wave BCS at the mean-field level: pairing
> instability, gap equation, thermodynamics, density of states, and
> the universal ratios. Where the attraction comes from
> (Migdal–Eliashberg theory), what strong coupling does to lead, and
> everything unconventional (d-wave cuprates, where
> [§8.13](hubbard-model.ipynb)'s doped Mott insulator returns) is
> Tinkham {cite}`tinkham2004` and the modern literature.

## Theory in brief

### Cooper's logarithm: the Fermi surface is unstable

Two electrons added at $\pm\mathbf k$ above a filled Fermi sea, with
attraction $-V$ acting in the shell $0 < \xi_k < \omega_D$, admit a
bound state whose energy $E_b > 0$ below $2E_F$ solves

```{math}
:label: eq-bcs-cooper
1 \;=\; \lambda \int_0^{\omega_D}
\frac{d\xi}{2\xi + E_b}
\;\;\Longrightarrow\;\;
E_b \;\xrightarrow{\lambda \to 0}\; 2\omega_D\,e^{-2/\lambda} ,
```

with $\lambda = N_0V$. The integral *diverges logarithmically* as
$E_b \to 0$: however weak the attraction, the equation always closes —
the Pauli-blocked continuum at the Fermi surface acts like
[§6.11](../06-quantum-mechanics/bound-states-1d.ipynb)'s
one-dimensional well, which also binds at any depth. And
$e^{-2/\lambda}$ has every derivative zero at $\lambda = 0$:
superconductivity is invisible to perturbation theory at any finite
order. The normal metal is not a stable starting point; it is a false
vacuum.

### The gap equation, at every temperature

BCS {cite}`bardeen1957` promoted the two-electron instability to a
self-consistent condensate of pairs. The variational ground state
leads to quasiparticles of energy $E = \sqrt{\xi^2 + \Delta^2}$ and
the finite-temperature self-consistency

```{math}
:label: eq-bcs-gap
1 \;=\; \lambda \int_0^{\omega_D} d\xi\,
\frac{\tanh\!\big(\sqrt{\xi^2 + \Delta^2}/2T\big)}
{\sqrt{\xi^2 + \Delta^2}} ,
```

whose $T = 0$ limit gives $\Delta_0 = \omega_D/\sinh(1/\lambda)$
exactly, and whose $\Delta \to 0$ limit defines $T_c =
(2e^\gamma/\pi)\,\omega_D e^{-1/\lambda} \approx 1.134\,\omega_D
e^{-1/\lambda}$. Same exponential, different prefactors — so the
ratio

```{math}
:label: eq-bcs-ratio
\frac{2\Delta_0}{k_B T_c} \;=\; \frac{2\pi}{e^\gamma} \;=\; 3.528
```

contains *no* material parameters at all: the theory's cleanest
falsifiable claim, and Exercise 6's courtroom exhibit.

### Thermodynamics and the density of states

The quasiparticles are fermions with a temperature-dependent
dispersion, so [§7.9](../07-quantum-statistical-mechanics/fermi-gas-zero-temperature.ipynb)'s
machinery applies verbatim: entropy $S = -2N_0\int d\xi\,[f\ln f +
(1-f)\ln(1-f)]$ (both spins, both signs of $\xi$), specific heat
$C = T\,dS/dT$ — which inherits a *jump* at $T_c$ from
$d\Delta^2/dT$, with the universal value $\Delta C/C_n = 12/7\zeta(3)
= 1.426$, and an exponential freeze-out below (the gap gating
excitations as a semiconductor's does). The one-to-one mapping
$E = \sqrt{\xi^2+\Delta^2}$ also reshuffles the density of states
into

```{math}
:label: eq-bcs-dos
\frac{N_s(E)}{N_0} \;=\;
\frac{E}{\sqrt{E^2 - \Delta^2}}\;\theta(E - \Delta) :
```

a hard gap plus square-root *coherence peaks*, with the states
conservation ledger in closed form
($\int_0^X (N_s/N_0 - 1)\,dE = \sqrt{X^2 - \Delta^2} - X$): what the
gap evicts, the peaks absorb. This is precisely the shape a
tunneling conductance measures, which is how $\Delta$ is read off
real materials.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import brentq
from scipy.special import zeta

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

OMEGA_D = 1.0  # the Debye shell: the energy unit
LAMBDA = 0.3  # default coupling N0 V


def gap_rhs(delta, temp, lam):
    """Right-hand side of the BCS gap equation, Eq. (eq-bcs-gap).

    Parameters
    ----------
    delta : float
        Trial gap.
    temp : float
        Temperature (k_B = 1).
    lam : float
        Coupling N0 V.

    Returns
    -------
    float
        lambda * integral; the gap equation reads rhs = 1.
    """
    integrand = lambda xi: (
        np.tanh(np.sqrt(xi**2 + delta**2) / (2.0 * temp)) / np.sqrt(xi**2 + delta**2)
    )
    return lam * quad(integrand, 0.0, OMEGA_D, limit=200)[0]


def gap_at(temp, lam=LAMBDA):
    """Self-consistent gap Delta(T) by brentq on Eq. (eq-bcs-gap).

    Parameters
    ----------
    temp : float
        Temperature.
    lam : float, optional
        Coupling.

    Returns
    -------
    float
        The gap (0 above T_c).
    """
    residual = lambda d: gap_rhs(d, temp, lam) - 1.0
    if residual(1e-12) < 0.0:
        return 0.0
    return brentq(residual, 1e-12, 2.0 * OMEGA_D, xtol=1e-15)


def critical_temperature(lam=LAMBDA):
    """T_c from the Delta -> 0 limit of the gap equation.

    Parameters
    ----------
    lam : float, optional
        Coupling.

    Returns
    -------
    float
        The critical temperature.
    """
    return brentq(lambda t: gap_rhs(1e-14, t, lam) - 1.0, 1e-6, 2.0 * OMEGA_D)

## Exercise 1 — Cooper's instability: bound at any coupling

The two-electron problem that broke the 46-year logjam.

**Part a)** Solve Eq. {eq}`eq-bcs-cooper` for the pair binding energy
with `scipy.optimize.brentq` at $\lambda = 0.1$–$0.5$ (the integral is
elementary: $1 = \tfrac\lambda2\ln[(2\omega_D + E_b)/E_b]$). A bound
state exists at *every* coupling — at $\lambda = 0.1$ the binding is
$4\times10^{-9}\,\omega_D$, absurdly small and stubbornly nonzero.

**Part b)** Verify the weak-coupling law: $E_b/(2\omega_D
e^{-2/\lambda})$ must approach $1$ from above as $\lambda$ shrinks
(measured $1.019 \to 1.001 \to 1.000$ at $\lambda = 0.5, 0.3, 0.2$;
below that the binding sits at the root-finder's precision floor and
the ratio simply pins to $1$).
Then make the theoretical point *graphically*: $\ln E_b$ against
$1/\lambda$ is a straight line of slope $-2$ — a function whose every
Taylor coefficient at $\lambda = 0$ vanishes, invisible to any
perturbative expansion in the interaction. The instability had to be
found nonperturbatively, and was.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the essential singularity, measured

The weak-coupling ratio at $10^{-3}$ by $\lambda = 0.2$, approaching
$1$ monotonically; the semilog slope $-2$.

In [ ]:
validate.close(
    float(ratios[2]), 1.0, "E_b -> 2 wD e^(-2/lambda) (lambda = 0.2)", rtol=1e-3
)
validate.check(
    bool(
        np.all(np.diff(ratios[2:]) > 0.0) and np.all((ratios > 0.999) & (ratios < 1.05))
    ),
    "weak-coupling law approached from above (monotone for lambda >= 0.2)",
    f"ratios {ratios[2]:.4f} -> {ratios[-1]:.4f}",
)
validate.close(slope, -2.0, "ln E_b linear in 1/lambda with slope -2", rtol=1e-2)

## Exercise 2 — The gap at zero temperature

From one bound pair to a condensate of them.

**Part a)** Solve the gap equation at $T \to 0$ numerically for
$\lambda = 0.25, 0.3, 0.4$ and check the exact closed form
$\Delta_0 = \omega_D/\sinh(1/\lambda)$ at $10^{-9}$ — the $T = 0$
integral is $\lambda\,\mathrm{asinh}(\omega_D/\Delta)$, and `brentq`
against `quad` must agree with algebra.

**Part b)** Note the structural rhyme and its factor-of-two
difference: $\Delta_0 \to 2\omega_D e^{-1/\lambda}$ at weak coupling —
the *same* essential singularity as Cooper's $E_b$, but with
$e^{-1/\lambda}$ in place of $e^{-2/\lambda}$: the many-body
condensate binds exponentially *more* strongly than one pair atop a
frozen sea (measured: $\Delta_0/E_b = 28$ at $\lambda = 0.3$).
Cooperation, literally.

In [ ]:
# (solution hidden on the public site)


### Validation 2 — algebra against quadrature

The closed form at $10^{-9}$; the condensate's exponential advantage.

In [ ]:
validate.check(
    bool(gap_dev < 1e-8),
    "Delta_0 = wD / sinh(1/lambda) at three couplings",
    f"worst dev {gap_dev:.1e}",
)
validate.check(
    bool(20.0 < boost < 40.0),
    "the condensate outbinds the lone Cooper pair ~30-fold",
    f"ratio {boost:.1f}",
)

## Exercise 3 — $\Delta(T)$, $T_c$, and the universal ratio

The full temperature dependence, and the numbers that made BCS
falsifiable.

**Part a)** Solve Eq. {eq}`eq-bcs-gap` across $0 < T < T_c$ for
$\lambda = 0.25, 0.3, 0.4$ and plot $\Delta(T)/\Delta_0$ against
$T/T_c$: the three curves must *collapse onto one universal shape* —
BCS thermodynamics knows only the reduced variables.

**Part b)** Extract $T_c$ from the $\Delta \to 0$ limit and test the
closed form $T_c = (2e^\gamma/\pi)\,\omega_D e^{-1/\lambda}$: the
ratio must be $1.00000$ at all three couplings (five digits — the
numerics is solving the same equation the asymptotics solved).

**Part c)** Form the material-free ratio, Eq. {eq}`eq-bcs-ratio`:
$2\Delta_0/T_c = 3.529$ at $\lambda = 0.25$ against the universal
$2\pi e^{-\gamma} = 3.528$ (with a measured, monotone weak-coupling
drift: $3.532$ at $\lambda = 0.3$, $3.552$ at $0.4$ — the corrections
the ratio hides at stronger coupling, foreshadowing lead).

**Part d)** Fit the near-$T_c$ collapse: $\Delta(T)/T_c =
A\sqrt{1 - T/T_c}$ with the fitted $A = 3.04$ against BCS's $3.06$ —
the mean-field order-parameter exponent $\beta = \tfrac12$ of
[§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb), at the
volume's closing phase transition.

In [ ]:
# (solution hidden on the public site)


### Validation 3 — the falsifiable numbers

$T_c$ on its closed form at five digits; the universal ratio at the
weak-coupling point; the collapse coefficient; the monotone
strong-coupling drift.

In [ ]:
validate.check(
    bool(tc_dev < 1e-4),
    "T_c = 1.134 wD e^(-1/lambda) at three couplings (5 digits)",
    f"worst dev {tc_dev:.1e}",
)
validate.close(
    float(ratio_list[0]),
    universal,
    "2 Delta_0/T_c = 2 pi e^(-gamma) = 3.528",
    rtol=1e-3,
)
validate.check(
    bool(ratio_list[0] < ratio_list[1] < ratio_list[2]),
    "the ratio drifts up with coupling (the road to lead)",
    f"{ratio_list[0]:.4f} -> {ratio_list[2]:.4f}",
)
validate.check(
    bool(2.95 < coeff < 3.10),
    "Delta ~ 3.06 T_c sqrt(1 - t): mean-field beta = 1/2",
    f"fitted {coeff:.3f}",
)

## Exercise 4 — Thermodynamics: the jump and the freeze

The gap's fingerprints on the specific heat — historically the first
quantitative BCS confirmations.

**Part a)** Build the quasiparticle entropy per $N_0$
($S = 4\int_0^\infty[\,zf + \ln(1 + e^{-z})\,]d\xi$ with
$z = E/T$ — the softplus form, stable at low $T$ where
$f \ln f$ underflows) with $\Delta(T)$ *inside* the integrand, and
the specific heat $C = T\,dS/dT$ by central differences with steps
relative to $T_c$ (an absolute step straddles the transition and
destroys the jump — measured lesson).

**Part b)** The jump: extrapolate $C_s(T \to T_c^-)$ linearly from
$[0.98, 0.998]\,T_c$ and compare with the normal-state Sommerfeld
value $C_n = (2\pi^2/3)\,T_c$ (verified numerically at $0.3\%$):
$\Delta C/C_n = 1.43$, against the universal $12/7\zeta(3) = 1.426$.

**Part c)** The freeze: at $T = 0.2\,T_c$ the superconducting
specific heat is $3\%$ of the normal-state value at the same
temperature (a thirty-fold suppression) — the gap gating excitations exponentially, exactly as
the semiconductor gaps of
[§7.13](../07-quantum-statistical-mechanics/semiconductor-statistics.ipynb)
did, but this gap the electrons *built themselves*.

In [ ]:
# (solution hidden on the public site)


### Validation 4 — calorimetry agrees with algebra

Sommerfeld verified; the universal jump within $0.5\%$; the
exponential freeze-out.

In [ ]:
validate.close(
    c_normal_num, sommerfeld * T_C, "normal-state Sommerfeld heat", rtol=5e-3
)
validate.close(
    jump, float(12.0 / (7.0 * zeta(3))), "the universal jump 12/(7 zeta(3))", atol=0.02
)
validate.check(
    bool(freeze < 0.04),
    "excitations frozen out at 0.2 T_c (< 4% of normal)",
    f"C_s/C_n = {freeze:.4f}",
)

## Exercise 5 — The density of states: what tunneling sees

The gap is not just an energy scale; it is a hole carved into the
spectrum, with the evicted states piled at its edges.

**Part a)** Plot Eq. {eq}`eq-bcs-dos` for $\Delta = 0.2\,\omega_D$:
hard gap below $\Delta$, divergent coherence peaks just above —
integrable ($\sqrt{E - \Delta}$-type), so the peaks are tall but
carry finite weight.

**Part b)** Audit the ledger in closed form: states are conserved
under the $\xi \to E$ mapping, and the running deficit obeys exactly
$\int_0^X (N_s/N_0 - 1)\,dE = \sqrt{X^2 - \Delta^2} - X$ — negative
at any finite $X$ (the gap's evicted states are recovered from above
only asymptotically, as $-\Delta^2/2X$). `quad` with a declared singular
point must match the closed form at $10^{-6}$: the same
weight-conservation bookkeeping every sum rule in this volume
enforced, one last time.

In [ ]:
# (solution hidden on the public site)


### Validation 5 — the books balance, in closed form

Quadrature against algebra at $10^{-6}$; the deficit negative and
asymptotically $-\Delta^2/2X$.

In [ ]:
validate.close(
    numeric_deficit, closed_deficit, "state ledger = sqrt(X^2-D^2) - X", rtol=1e-6
)
validate.close(
    closed_deficit,
    float(-(DELTA_DOS**2) / (2.0 * x_cut)),
    "deficit -> -Delta^2/2X asymptotically",
    rtol=2e-3,
)

## Exercise 6 — Verdict: the elements take the stand

A universal number is a hostage to experiment. Compare Eq.
{eq}`eq-bcs-ratio` with measured gap-to-$T_c$ ratios from tunneling
spectroscopy {cite}`tinkham2004`.

**Part a)** The weak-coupling elements: aluminum ($2\Delta/k_BT_c =
3.4$), tin ($3.5$), indium ($3.6$) — deviations $-3.6\%$, $-0.8\%$,
$+2.0\%$ from $3.528$. Three different metals, three different
$T_c$'s spanning a factor of three, one parameter-free prediction
within $4\%$: this is what made BCS irresistible.

**Part b)** The honest exception: lead, at $4.38$ — $+24\%$. Lead's
electron–phonon coupling is strong ($\lambda \approx 1.5$) and its
phonons slow; the instantaneous-attraction idealization of Eq.
{eq}`eq-bcs-gap` breaks, exactly the direction Exercise 3's
strong-coupling drift pointed. (Migdal–Eliashberg theory, which keeps
the retardation, gets lead right — the fix is known physics, not
folklore.) Print the course's final verdict table and close the
volume: an attraction of order *millielectronvolts*, mediated by
lattice vibrations, self-organizing into a macroscopic quantum state
that carries current without loss — computed, gated, and checked
against five elements.

In [ ]:
# (solution hidden on the public site)


### Validation 6 — within 4%, and honestly outside it

The weak-coupling trio inside $4\%$; lead outside $20\%$ — both
facts gated, because both are the lesson.

In [ ]:
validate.check(
    bool(all(abs(devs[n]) < 0.04 for n in WEAK)),
    "Al, Sn, In within 4% of the universal ratio",
    ", ".join(f"{n} {100*devs[n]:+.1f}%" for n in WEAK),
)
validate.check(
    bool(devs["Pb"] > 0.20 and devs["Nb"] > 0.05),
    "Pb (and Nb) above the ratio: strong coupling flagged, not hidden",
    f"Pb {100*devs['Pb']:+.1f}%, Nb {100*devs['Nb']:+.1f}%",
)

:::{admonition} With your assistant
:class: tip
The gap equation holds more thermodynamics: the *condensation energy*
— how much the pairs' organization is worth. Have your assistant
compute $F_n - F_s$ at $T \to 0$ by integrating the gap equation's
coupling dependence (or directly: $E_{\mathrm{cond}} =
\tfrac12 N_0\Delta_0^2$ for weak coupling) and convert it, for
aluminum ($\Delta_0 = 0.18$ meV, $N_0 \approx 12\ \mathrm{eV}^{-1}$
per atom... look the numbers up), into kelvin per atom. Then run the
check that is yours alone: the condensation energy per atom must come
out *five or more orders of magnitude below* $k_BT_c$ per electron
times... no — compare it against the Fermi energy: the ratio
$E_{\mathrm{cond}}/E_F$ must be of order $(\Delta_0/E_F)^2 \sim
10^{-9}$ (`numpy` arithmetic, order-of-magnitude `assert`). A
macroscopic quantum state held together by one part in a billion of
the electronic energy: fragility and robustness at once. The check is
yours.
:::

## Notebook summary

The volume's last computation gave the interaction map its missing
corner. Cooper's two-electron problem closed at *every* coupling —
binding $2\omega_D e^{-2/\lambda}$ reproduced at $10^{-3}$, semilog
slope $-2.00$ — an essential singularity that no perturbative order
sees, which is why the phenomenon outran theory for 46 years. The
full gap equation, solved by `brentq` at every temperature, hit
$\Delta_0 = \omega_D/\sinh(1/\lambda)$ at $10^{-9}$ and
$T_c = 1.134\,\omega_D e^{-1/\lambda}$ at five digits, collapsed
three couplings onto one universal $\Delta(T)/\Delta_0$ curve with
the mean-field $3.04\sqrt{1 - T/T_c}$ die-off, and delivered the
parameter-free $2\Delta_0/k_BT_c = 3.528$. The thermodynamics
followed from a numerical entropy with the gap self-consistently
inside: the universal jump $\Delta C/C_n = 1.43$ against
$12/7\zeta(3) = 1.426$, the Sommerfeld normal state at $0.3\%$, and
a $3\%$ freeze-out at $0.2\,T_c$. The density of states carved its
gap and paid for it in coherence peaks, ledger in closed form at
$10^{-6}$. And the elements ruled: aluminum, tin, indium within
$4\%$ of the universal ratio; lead $24\%$ high and *named* as the
strong-coupling exception it is. From [§8.1](many-electron-problem.ipynb)'s
impossible wavefunction to a macroscopic quantum state carrying
current without loss — every step of the road computed, and every
claim on it gated.

## Outlook

- Where the attraction comes from — and why lead misbehaves — is
  Migdal–Eliashberg theory: the phonon retardation kept honestly,
  [§8.14](quasiparticles-gw.ipynb)'s self-energy machinery with a
  phonon propagator inside. Tinkham {cite}`tinkham2004` carries the
  full story, Ginzburg–Landau theory included (whose $\beta = \tfrac12$
  this notebook already met).
- The cuprate superconductors start from
  [§8.13](hubbard-model.ipynb)'s doped Mott insulator, pair with
  d-wave symmetry, and remain — four decades on — the open problem
  this volume's tools were built to besiege.
- This closes Volume VIII, the last of the course's physics volumes.
  What remains is the [Epilogue](../epilogue/index.md) — four notebooks
  that turn the course's instruments on the course itself — and then
  the [Afterword](../epilogue/afterword.md) has the last word. It has
  been, from the falling projectile of
  [§1.1](../01-elementary-mechanics/projectile-drag.ipynb) to the
  condensate, a single story: write the physics, discretize it
  honestly, and *check everything*.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()